# 03 — cosine similarity


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import precision_score
import warnings
warnings.filterwarnings("ignore")

AUDIO_FEATURES = [
    "danceability", "energy", "loudness", "speechiness",
    "acousticness", "instrumentalness", "liveness", "valence", "tempo"
]
PROCESSED_DIR  = "../data/processed"
SPLIT_DIR      = "../outputs/train_test_split"
OUTPUTS_DIR    = "../outputs"
PLOTS_DIR      = f"{OUTPUTS_DIR}/plots"
os.makedirs(PLOTS_DIR, exist_ok=True)

## 3.1 — Load Data

In [ ]:
songs_train = pd.read_csv(f"{SPLIT_DIR}/songs_train.csv")
songs_test  = pd.read_csv(f"{SPLIT_DIR}/songs_test.csv")

print(f"Train: {len(songs_train):,} rows")
print(f"Test:  {len(songs_test):,} rows")
print(f"Columns: {list(songs_train.columns)}")
songs_train.head(3)

## 3.2 — Feature Normalization

`loudness` lives in [-60, 0] dB and `tempo` in [0, ~250] BPM while all other features are already in [0, 1].  
We re-apply `MinMaxScaler` **fit on the training set only** to put every feature in [0, 1] before computing cosine distance.

In [ ]:
scaler = MinMaxScaler()
X_train = scaler.fit_transform(songs_train[AUDIO_FEATURES])
X_test  = scaler.transform(songs_test[AUDIO_FEATURES])

print(f"X_train shape: {X_train.shape}")
print(f"X_test  shape: {X_test.shape}")

# Quick sanity check — all values should be in [0, 1]
print(f"Train min={X_train.min():.4f}, max={X_train.max():.4f}")
print(f"Test  min={X_test.min():.4f},  max={X_test.max():.4f}")

## 3.3 — Build Cosine Similarity Model

We use `sklearn.neighbors.NearestNeighbors` with `metric='cosine'`.  
This avoids materialising the full 557K × 557K similarity matrix (~1.2 TB) — instead it computes distances on demand per query in O(n · d) time.

In [ ]:
nn_model = NearestNeighbors(metric="cosine", algorithm="brute", n_jobs=-1)
nn_model.fit(X_train)
print("NearestNeighbors model fitted on", X_train.shape[0], "songs.")

## 3.4 — Song Recommendation Function

In [ ]:
def recommend_similar_songs(query_track_name, query_artist_name=None, top_k=10):
    """
    Find top_k songs most similar to a query track using cosine similarity
    on normalized audio features.

    Looks up the query in songs_train first; falls back to songs_test.
    Returns a DataFrame of recommendations with cosine similarity scores.
    """
    # Locate query song
    mask = songs_train["track_name"].str.lower() == query_track_name.lower()
    if query_artist_name:
        mask &= songs_train["artist_name"].str.lower() == query_artist_name.lower()
    
    source_df, source_X = songs_train, X_train
    if mask.sum() == 0:
        mask = songs_test["track_name"].str.lower() == query_track_name.lower()
        if query_artist_name:
            mask &= songs_test["artist_name"].str.lower() == query_artist_name.lower()
        source_df, source_X = songs_test, X_test

    if mask.sum() == 0:
        print(f"Song '{query_track_name}' not found.")
        return None

    query_idx   = source_df[mask].index[0]
    query_row   = source_df.loc[query_idx]
    query_vec   = source_X[source_df.index.get_loc(query_idx)].reshape(1, -1)

    # k+1 because the query itself may be in the training set
    distances, indices = nn_model.kneighbors(query_vec, n_neighbors=top_k + 1)

    results = []
    for dist, idx in zip(distances[0], indices[0]):
        row = songs_train.iloc[idx]
        # Skip if it's the exact same song
        if row["track_name"].lower() == query_track_name.lower():
            continue
        results.append({
            "track_name":       row["track_name"],
            "artist_name":      row["artist_name"],
            "genre":            row["genre"],
            "popularity":       row["popularity"],
            "cosine_similarity": round(1 - dist, 4),
        })
        if len(results) == top_k:
            break

    print(f"Query: '{query_row['track_name']}' — {query_row['artist_name']}  |  genre: {query_row['genre']}")
    print("-" * 80)
    return pd.DataFrame(results)

## 3.5 — Demo: Find Similar Songs

In [ ]:
# Example 1 — Pop track
recs1 = recommend_similar_songs("Shape of You", top_k=10)
recs1

In [ ]:
# Example 2 — Hip-Hop track
recs2 = recommend_similar_songs("HUMBLE.", top_k=10)
recs2

In [ ]:
# Example 3 — Classical / low-energy track
recs3 = recommend_similar_songs("Clair de Lune", top_k=10)
recs3

In [ ]:
# Visualise audio feature profiles: query vs. top-5 recommendations
def plot_feature_radar(query_name, recs_df, n_recs=5):
    cats   = AUDIO_FEATURES
    N      = len(cats)
    angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))

    # Query song vector (from training set, already normalized)
    mask = songs_train["track_name"].str.lower() == query_name.lower()
    if mask.sum() == 0:
        mask = songs_test["track_name"].str.lower() == query_name.lower()
        q_vec = X_test[songs_test.index.get_loc(songs_test[mask].index[0])]
    else:
        q_vec = X_train[songs_train.index.get_loc(songs_train[mask].index[0])]

    vals = q_vec.tolist() + [q_vec[0]]
    ax.plot(angles, vals, linewidth=2, label=query_name)
    ax.fill(angles, vals, alpha=0.15)

    for _, row in recs_df.head(n_recs).iterrows():
        m = songs_train["track_name"] == row["track_name"]
        if m.sum() == 0:
            continue
        rv = X_train[songs_train.index.get_loc(songs_train[m].index[0])].tolist()
        rv += [rv[0]]
        ax.plot(angles, rv, linewidth=1, linestyle="--", alpha=0.7,
                label=row["track_name"][:30])

    ax.set_thetagrids(np.degrees(angles[:-1]), cats, fontsize=9)
    ax.set_title(f"Audio Profile — '{query_name}'", pad=20, fontsize=12)
    ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.15), fontsize=8)
    plt.tight_layout()
    plt.savefig(f"{PLOTS_DIR}/cosine_radar_{query_name[:20].replace(' ','_')}.png", dpi=120)
    plt.show()

plot_feature_radar("Shape of You", recs1)

## 3.6 — Evaluation

We use **genre-based Precision@K** as a proxy metric: for each test song that has a known genre, we query the model and measure what fraction of the top-K recommendations share the same genre.

A high Precision@K means the model consistently groups songs with the same sonic characteristics — a good signal for content-based filtering quality.

In [ ]:
def genre_precision_at_k(k=10, sample_size=2000, seed=42):
    """
    For a random sample of test songs with a known genre, compute the fraction
    of top-k recommendations (from training set) that share the same genre.
    Returns mean Precision@K and per-genre breakdown.
    """
    rng = np.random.default_rng(seed)
    test_with_genre = songs_test.dropna(subset=["genre"]).reset_index(drop=True)
    X_test_genre    = X_test[songs_test.dropna(subset=["genre"]).index - songs_test.index[0]]

    n = min(sample_size, len(test_with_genre))
    sample_idx = rng.choice(len(test_with_genre), size=n, replace=False)

    query_vecs   = X_test_genre[sample_idx]
    query_genres = test_with_genre.iloc[sample_idx]["genre"].values

    distances, indices = nn_model.kneighbors(query_vecs, n_neighbors=k)

    train_genres = songs_train["genre"].values
    precision_scores = []
    genre_hits = {}

    for i, (nbr_idxs, q_genre) in enumerate(zip(indices, query_genres)):
        rec_genres   = train_genres[nbr_idxs]
        hits         = (rec_genres == q_genre).sum()
        p_at_k       = hits / k
        precision_scores.append(p_at_k)
        genre_hits.setdefault(q_genre, []).append(p_at_k)

    mean_p = np.mean(precision_scores)
    genre_mean = {g: np.mean(v) for g, v in genre_hits.items() if len(v) >= 5}
    return mean_p, genre_mean

mean_p10, genre_breakdown = genre_precision_at_k(k=10, sample_size=2000)
print(f"Mean Genre Precision@10 : {mean_p10:.4f}  ({mean_p10*100:.1f}%)")

In [ ]:
# Precision@K curve: how does precision change as K grows?
ks = [1, 5, 10, 20, 50]
p_at_ks = []
for k in ks:
    p, _ = genre_precision_at_k(k=k, sample_size=1000)
    p_at_ks.append(p)
    print(f"  Precision@{k:2d} = {p:.4f}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ks, p_at_ks, marker="o", linewidth=2)
ax.set_xlabel("K")
ax.set_ylabel("Mean Genre Precision@K")
ax.set_title("Content-Based Cosine Similarity — Precision@K")
ax.set_xticks(ks)
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/cosine_precision_at_k.png", dpi=120)
plt.show()

In [ ]:
# Per-genre Precision@10 bar chart
genre_df = (pd.Series(genre_breakdown)
              .sort_values(ascending=False)
              .head(20)
              .reset_index())
genre_df.columns = ["genre", "precision_at_10"]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(genre_df["genre"], genre_df["precision_at_10"], color="steelblue")
ax.set_xlabel("Mean Precision@10")
ax.set_title("Genre-Level Precision@10 (Top 20 genres)")
ax.axvline(mean_p10, color="red", linestyle="--", label=f"Overall mean = {mean_p10:.3f}")
ax.legend()
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/cosine_precision_by_genre.png", dpi=120)
plt.show()

## 3.7 — Save Outputs

In [ ]:
import joblib

MODELS_DIR = f"{OUTPUTS_DIR}/models"
os.makedirs(MODELS_DIR, exist_ok=True)

# Save the fitted scaler and NearestNeighbors model for reuse in later notebooks
joblib.dump(scaler,   f"{MODELS_DIR}/cosine_scaler.pkl")
joblib.dump(nn_model, f"{MODELS_DIR}/cosine_nn_model.pkl")

# Save precision summary
precision_summary = pd.DataFrame({"k": ks, "precision_at_k": p_at_ks})
precision_summary.to_csv(f"{OUTPUTS_DIR}/cosine_precision_summary.csv", index=False)

print("Saved:")
print(f"  {MODELS_DIR}/cosine_scaler.pkl")
print(f"  {MODELS_DIR}/cosine_nn_model.pkl")
print(f"  {OUTPUTS_DIR}/cosine_precision_summary.csv")
print(f"  {PLOTS_DIR}/cosine_precision_at_k.png")
print(f"  {PLOTS_DIR}/cosine_precision_by_genre.png")